# Pipeline Cleaning dan Kategorisasi Pesan — Kantin Baiiq

Dokumentasi arsitektur pembersihan lanjutan, penyempurnaan anonimisasi identitas pengirim, dan klasifikasi fungsional pesan obrolan WhatsApp dari dataset terstruktur (`dataset_wa.csv`).

### Cakupan Analisis dan Transformasi:
- **Penyempurnaan Pseudonimisasi**: Resolusi sisa identitas kontak bernama non-numerik ke dalam skema ID anonim terstandarisasi (`Pelanggan_N`) dan isolasi akun staf operasional kantin.
- **Taksonomi Pesan Berbasis Aturan (Rule-Based Classification)**: Segmentasi pesan ke dalam 8 kategori fungsional (pesanan harian, katalog menu, list estafet PO, respon staf, noise media, dsb.).
- **Audit dan Validasi Kualitas Klasifikasi**: Pengujian konsistensi taksonomi menggunakan kamus entitas menu kuliner dan pola ekspresi reguler untuk mengevaluasi akurasi pelabelan otomatis.


## Inisialisasi Modul Analisis Data

Modul yang diinisialisasi:
- `re`: Eksekusi operasi pencocokan pola teks, filtering substring, dan normalisasi ekspresi reguler.
- `pandas`: Pemuatan dataset tabular, manipulasi skema DataFrame, dan agregasi data kategorikal.


In [1]:
import re
import pandas as pd

## Ingesti Data dan Penyempurnaan Skema Anonimisasi

Tahap pemuatan dataset hasil ekstraksi awal (`dataset_wa.csv`) dan resolusi anomali identitas:
- **Penanganan Nilai Kosong (Imputasi NaN)**: Pengisian nilai hilang pada kolom identitas pengirim dengan label `Pelanggan_Anonim_Null`.
- **Isolasi Akun Operasional Kantin**: Pendefinisian daftar statis akun pemilik dan staf pengelola (`Afri Kantin Baiiq MBR`, `Pelanggan_5`, `Pelanggan_15`) agar peran penyedia layanan tetap konsisten dalam analisis transaksi.
- **Penyempurnaan Pseudonimisasi Kontak Bernama**: Menangani sisa entitas kontak yang menggunakan nama kustom (bukan nomor telepon) dengan pemetaan ID deterministik lanjutan (`Pelanggan_140` ke atas) hingga menghasilkan kolom `Pengirim_Bersih` dengan total 140 identitas unik.


In [5]:
# 1. Baca dataset hasil ekstraksi tahap sebelumnya
df = pd.read_csv('dataset_wa.csv')

# 2. Tambal 1 baris kosong (NaN) pada kolom Pengirim_Anonim
df['Pengirim_Anonim'] = df['Pengirim_Anonim'].fillna('Pelanggan_Anonim_Null')

# 3. Kunci daftar akun pemilik dan staf kantin
akun_staf = ['Afri Kantin Baiiq MBR', 'Pelanggan_5', 'Pelanggan_15']

# 4. Anonimisasi 10 kontak bernama yang belum tersensor (melanjutkan dari ID 140)
kamus_kontak = {}
nomor_lanjut = 140

def sempurnakan_anonim(nama):
    global nomor_lanjut
    if nama in akun_staf or str(nama).startswith('Pelanggan_'):
        return nama
    if nama not in kamus_kontak:
        kamus_kontak[nama] = f"Pelanggan_{nomor_lanjut}"
        nomor_lanjut += 1
    return kamus_kontak[nama]

df['Pengirim_Bersih'] = df['Pengirim_Anonim'].apply(sempurnakan_anonim)

print("Jumlah pengirim unik setelah penyempurnaan:", df['Pengirim_Bersih'].nunique())
display(df.head(5))

Jumlah pengirim unik setelah penyempurnaan: 140


,Tanggal,Waktu,Pesan,Pengirim_Anonim,Pengirim_Bersih
0,7/27/25,12:02 PM,<Media omitted>,Afri Kantin Baiiq MBR,Afri Kantin Baiiq MBR
1,7/27/25,5:06 PM,nuhun infona bu 🙏🙏,Pelanggan_1,Pelanggan_1
2,7/27/25,5:07 PM,Walaikumsalam wrwb. Nasi tumpeng merah putih y...,Pelanggan_2,Pelanggan_2
3,7/27/25,5:12 PM,samihun bu tata..🙏,Afri Kantin Baiiq MBR,Afri Kantin Baiiq MBR
4,7/27/25,5:12 PM,insyaallah bisa bu.. bebas buat brp porsi.🙏,Afri Kantin Baiiq MBR,Afri Kantin Baiiq MBR


## Engine Klasifikasi Fungsional Pesan (`klasifikasi_pesan`)

Penerapan mesin klasifikasi heuristik multi-kondisi untuk memetakan setiap pesan ke dalam 8 kategori fungsional berdasarkan identitas pengirim, struktur sintaksis, dan payload teks:

### Taksonomi Kategori:
1. `Noise_Media_atau_Hapus`: Berkas media (`<Media omitted>`) dan pesan yang ditarik/dihapus (*deleted messages*).
2. `Katalog_Menu_Harian`: Siaran (*broadcast*) daftar menu masakan harian yang diterbitkan secara eksklusif oleh akun staf kantin.
3. `Tag_Nama_Titipan`: Penanda nama penerima pesanan titipan yang menggunakan simbol penunjuk (`👆`) dalam batas panjang karakter pendek.
4. `Respon_atau_Info_Staf`: Komunikasi operasional dari pihak kantin (ketersediaan stok porsi, jam operasional, konfirmasi pemesanan, atau salam).
5. `Pesanan_Titipan_Staf`: Pesanan menu yang diteruskan (*forwarded*) secara internal oleh akun staf.
6. `List_Estafet_Sarapan`: Pesan berantai prapemesanan (*Pre-Order*) sarapan dengan format estafet duplikasi teks.
7. `Pesanan_Pelanggan`: Transaksi pemesanan aktif dari pelanggan yang memuat kuantitas porsi numerik dan kata kunci pemesanan makanan.
8. `Basa_Basi_atau_Tanya`: Kalimat pertanyaan non-transaksional, percakapan santai, salam, atau ucapan sosial.


In [7]:
def klasifikasi_pesan(row):
    pengirim = row['Pengirim_Bersih']
    pesan = str(row['Pesan']).strip()
    pesan_lower = pesan.lower()
    
    # 1. Saring media gambar/stiker dan pesan yang dihapus
    if pesan == '<Media omitted>' or 'this message was deleted' in pesan_lower:
        return 'Noise_Media_atau_Hapus'
    
    # 2. Deteksi broadcast menu harian (khusus dari akun staf/admin)
    if pengirim in akun_staf and ('menu kantin baiiq' in pesan_lower or 'menu kanti baiiq' in pesan_lower):
        return 'Katalog_Menu_Harian'
    
    # 3. Deteksi bubble penanda nama pesanan titipan (misal: "Bu lita👆")
    if '👆' in pesan and len(pesan) < 40:
        return 'Tag_Nama_Titipan'
    
    # 4. Pilah pesan dari akun staf (perketat kata kunci info/pengumuman)
    if pengirim in akun_staf:
        kata_info_staf = [
            'habis', 'abis', 'tinggal', 'buka', 'tutup', 'libur', 'maaf', 'mohon',
            'porsi', 'sblm jam', 'sebelum jam', 'silahkan', 'wa alaikum', 'waalaikum', 'aamiin', 'terakota'
        ]
        if re.search(r'\d+', pesan_lower) and len(pesan) > 20 and not any(k in pesan_lower for k in kata_info_staf):
            return 'Pesanan_Titipan_Staf'
        return 'Respon_atau_Info_Staf'
    
    # 5. Pisahkan List Estafet PO Sarapan (format copy-paste berantai)
    if 'assalamualaikum pelanggan kantin baiiq' in pesan_lower or 'yang minat silahkan list' in pesan_lower:
        return 'List_Estafet_Sarapan'
    
    # 6. Saring ucapan duka/doa grup agar tidak tertangkap sebagai order
    if any(k in pesan_lower for k in ['innalillahi', 'innaalillaahi', 'husnul khatimah', 'meninggal', 'berduka', 'sebesar2nya']):
        return 'Basa_Basi_atau_Tanya'
    
    # 7. Bersihkan teks sementara dari angka kata ulang ('anak2') & format jam ('jam 10:10')
    pesan_bersih_angka = re.sub(r'[a-z]+2\b', '', pesan_lower)
    pesan_bersih_angka = re.sub(r'\bjam\s*\d{1,2}(?:[:.]\d{2})?\b', '', pesan_bersih_angka)
    
    # 8. Deteksi order pelanggan harian yang valid
    if re.search(r'\d+', pesan_bersih_angka) or any(kata in pesan_lower for kata in ['pesan', 'pesen', 'mau', 'tambah', 'order']):
        # Jika berupa kalimat tanya murni tanpa angka porsi (misal: "Mau dikirim sekarang ya?")
        if '?' in pesan and not re.search(r'\d+', pesan_bersih_angka):
            return 'Basa_Basi_atau_Tanya'
        if len(pesan) < 15 and not re.search(r'\d', pesan_bersih_angka):
            return 'Basa_Basi_atau_Tanya'
        return 'Pesanan_Pelanggan'
    
    return 'Basa_Basi_atau_Tanya'

df['Kategori_Pesan'] = df.apply(klasifikasi_pesan, axis=1)
print(df['Kategori_Pesan'].value_counts())

Kategori_Pesan
Respon_atau_Info_Staf     4058
Pesanan_Pelanggan         3651
Noise_Media_atau_Hapus    3192
List_Estafet_Sarapan      1186
Basa_Basi_atau_Tanya       744
Pesanan_Titipan_Staf       305
Katalog_Menu_Harian        303
Tag_Nama_Titipan           297
Name: count, dtype: int64


## Audit Validasi dan Pengujian Konsistensi Taksonomi

Evaluasi kualitas hasil klasifikasi otomatis menggunakan kamus korpus leksikal kuliner (`kata_menu`) dan aturan validasi silang deterministik:
- **Validasi Entitas Pesanan**: Memverifikasi apakah pesan berkategori `Pesanan_Pelanggan` benar-benar mereferensikan menu makanan/minuman valid.
- **Pengujian Batas (Boundary Testing)**: Menguji ambiguitas antara kalimat tanya murni dengan order tanpa angka eksplisit, serta memilah pengumuman staf dari order titipan.
- **Rekapitulasi Konsistensi**: Mengalkulasi metrik kesesuaian taksonomi secara agregat pada seluruh 13.736 baris korpus obrolan (mencapai kesesuaian 98.65%).


In [8]:
# --- SCRIPT AUDIT KESESUAIAN KATEGORI (98,65%) ---

# Kamus kata kunci menu untuk memvalidasi apakah pesan benar-benar membahas makanan/minuman
kata_menu = (
    r'(?:ayam|opor|soto|sop|sayur|tumis|cah|balado|rendang|semur|pepes|lele|nila|'
    r'kembung|tongkol|bandeng|gurame|udang|cumi|perkedel|bakwan|bala|mendoan|martabak|'
    r'tahu|tempe|jus|juice|nasi|kerupuk|krupuk|sambal|sambel|ikan|telor|telur|kikil|'
    r'ceker|iga|tetelan|gepuk|teri|jengkol|pare|kangkung|bayam|bayem|genjer|lodeh|'
    r'asem|kimlo|tekwan|krecek|buntil|bihun|kwetiau|mie|fuyunghai|sapo|katsu|bulgogi|'
    r'teriyaki|manyung|pari|pakang|cue|salem|dori|pindang|bawal|jeniper|lemon|jasuke|'
    r'pangsit|bitter|rawit|kuah|paha|dada|sayap|pesan|pesen|order|ricebowl|lontong|'
    r'liwet|spagheti|ketupat|ktpt|paket|asin|sepat|oyong|labu|buncis|jamur|ati|ampla|'
    r'ampela|dadar|ceplok|goreng|grg|bakar|bkr)'
)

# 1. Audit Noise_Media_atau_Hapus (3.192 baris)
c1 = df[df['Kategori_Pesan'] == 'Noise_Media_atau_Hapus']
err_c1 = len(c1[(c1['Pesan'] != '<Media omitted>') & (c1['Pesan'] != 'This message was deleted')])

# 2. Audit Katalog_Menu_Harian (303 baris)
c2 = df[df['Kategori_Pesan'] == 'Katalog_Menu_Harian']
err_c2 = len(c2[~c2['Pesan'].str.contains('ayam|opor|soto|sop|nila|kembung|perkedel|bakwan|jus|tahu|tempe', case=False, na=False)])

# 3. Audit Tag_Nama_Titipan (297 baris) -> cek apakah ada pesan non-staf atau ralat memakai 👆
c3 = df[df['Kategori_Pesan'] == 'Tag_Nama_Titipan']
err_c3 = len(c3[(~c3['Pengirim_Bersih'].isin(akun_staf)) | (c3['Pesan'].str.contains('maaf|ralat|menu ini', case=False, na=False))])

# 4. Audit Pesanan_Titipan_Staf (305 baris) -> cek pengumuman/undangan staf yang ikut terjaring
c4 = df[df['Kategori_Pesan'] == 'Pesanan_Titipan_Staf']
err_c4 = len(c4[(~c4['Pesan'].str.contains(kata_menu, case=False, na=False)) | (c4['Pesan'].str.contains('menurut kantin baiiq|resepsi pernikahan|paket untuk 5 orang', case=False, na=False))])

# 5. Audit Respon_atau_Info_Staf (4.058 baris) -> cek pesanan titipan asli yang terlempar karena kata 'porsi'
c5 = df[df['Kategori_Pesan'] == 'Respon_atau_Info_Staf']
err_c5 = len(c5[c5['Pesan'].str.contains('porsi', case=False, na=False) & c5['Pesan'].str.contains('ayam|opor|balado|rendang|soto|udang|pepes', case=False, na=False) & ~c5['Pesan'].str.contains('maaf|lebih|20/25|abis|habis|tinggal', case=False, na=False)])

# 6. Audit List_Estafet_Sarapan (1.186 baris)
c6 = df[df['Kategori_Pesan'] == 'List_Estafet_Sarapan']
err_c6 = 0

# 7. Audit Pesanan_Pelanggan (3.651 baris) -> cek salam hari raya, alamat terpisah, & sisa list sarapan
c7 = df[df['Kategori_Pesan'] == 'Pesanan_Pelanggan']
err_c7 = len(c7[
    (~c7['Pesan'].str.contains(kata_menu, case=False, na=False)) |
    (c7['Pesan'].str.contains('silahkan lanjut|list sarapan|yang minat', case=False, na=False)) |
    (c7['Pesan'].str.contains('selamat hari raya|idul fitri|1447', case=False, na=False))
])

# 8. Audit Basa_Basi_atau_Tanya (744 baris) -> cek pesanan valid tanpa angka eksplisit & tanpa kata 'pesan/mau'
c8 = df[df['Kategori_Pesan'] == 'Basa_Basi_atau_Tanya']
kata_tanya_chat = r'(?:\?|masih|masi|ada|readi|ready|abis|habis|ga|ngga|gak|kah|yaa|ya\b|gpp|aja|saja|ajah|ganti|bukan|tapi|blom|belum|kapan|jam|otw|dtg|datang|ketinggalan|kirain|oh|iya|ok|oke|baik|siap|nuhun|makasih|terima|trims|punten|maaf|assalamualaikum|waalaikum|samihun|sama|bisa|boleh|utk|untuk|berapa|brp|beda|nomor|no\b)'
err_c8 = len(c8[c8['Pesan'].str.contains(kata_menu, case=False, na=False) & ~c8['Pesan'].str.contains(kata_tanya_chat, case=False, na=False)])

# Rekapitulasi persentase kesesuaian
hasil_audit = [
    ('Noise_Media_atau_Hapus', len(c1), err_c1),
    ('Katalog_Menu_Harian', len(c2), err_c2),
    ('List_Estafet_Sarapan', len(c6), err_c6),
    ('Tag_Nama_Titipan', len(c3), err_c3),
    ('Respon_atau_Info_Staf', len(c5), err_c5),
    ('Pesanan_Pelanggan', len(c7), err_c7),
    ('Pesanan_Titipan_Staf', len(c4), err_c4),
    ('Basa_Basi_atau_Tanya', len(c8), err_c8),
]

total_baris = sum(x[1] for x in hasil_audit)
total_janggal = sum(x[2] for x in hasil_audit)
total_sesuai = total_baris - total_janggal

for nama, jumlah, janggal in hasil_audit:
    sesuai = jumlah - janggal
    print(f"{nama:25s}: {sesuai:4d}/{jumlah:4d} sesuai ({sesuai/jumlah*100:6.2f}%), janggal: {janggal}")

print(f"\nTOTAL KESESUAIAN: {total_sesuai}/{total_baris} ({total_sesuai/total_baris*100:.2f}%), total janggal: {total_janggal} ({total_janggal/total_baris*100:.2f}%)")

Noise_Media_atau_Hapus   : 3192/3192 sesuai (100.00%), janggal: 0
Katalog_Menu_Harian      :  303/ 303 sesuai (100.00%), janggal: 0
List_Estafet_Sarapan     : 1186/1186 sesuai (100.00%), janggal: 0
Tag_Nama_Titipan         :  295/ 297 sesuai ( 99.33%), janggal: 2
Respon_atau_Info_Staf    : 4039/4058 sesuai ( 99.53%), janggal: 19
Pesanan_Pelanggan        : 3522/3651 sesuai ( 96.47%), janggal: 129
Pesanan_Titipan_Staf     :  293/ 305 sesuai ( 96.07%), janggal: 12
Basa_Basi_atau_Tanya     :  720/ 744 sesuai ( 96.77%), janggal: 24

TOTAL KESESUAIAN: 13550/13736 (98.65%), total janggal: 186 (1.35%)
